In [35]:
import numpy as np
import pandas as pd
import kagglehub
import numpy.linalg as linalg
import matplotlib.pyplot as plt

In [8]:
#Closed Form Linear Regression

def LinRegExact(X,y):
    mat=np.dot(linalg.inv(np.dot(X.transpose(),X)),X.transpose())
    beta=np.dot(mat,y)
    return beta

In [47]:
#Grad desc. Linear Regression Normalised

def LinRegGradDesc(X,y,learningRate=0.01,iterations=500000):
    test=numpy.random.normal(0,1,np.shape(X[1]))
    Grad= lambda test: -2*X.T@(y-X@test)
    RSS= lambda test: (y-X@test).T@(y-X@test)
    rss=np.zeros(iterations)
    for i in range(iterations):
        test-= 1/np.shape(X)[0]*Grad(test)*learningRate
        rss[i]=RSS(test)
    return test,rss

In [12]:
#Ridge Regression

def RidgeReg(A,y,lam=50):
    #Input data X has no ones column for the constant beta_0
    betaR=linalg.solve(A.T@A+lam*np.identity(np.shape(A)[1]), A.T@(y-np.mean(y)))
    beta0=np.mean(y)-np.mean(A,axis=0)@betaR
    beta2=np.append(beta0,betaR)
    return beta2

In [39]:
#Data preparation (adding 1s columns, coding gender and normalising data)

abaloneData=pd.read_csv('abalone.data.csv')
abaloneData['gender']=abaloneData['gender'].map({'M':1, 'I':0, 'F':-1}) #code genders into numbers
rings=np.array(abaloneData['Rings'])
originaldata=np.array(abaloneData.drop('Rings',axis=1))
mu=np.mean(originaldata)
sigma=np.std(originaldata)
data=(originaldata-mu)/sigma
X=np.column_stack((np.ones(np.shape(data)[0]),data))
corrmat=np.corrcoef(data,rowvar=False)

In [ ]:
#Coefficients
LSExact=LinRegExact(X,rings)
LSGD,trainingRSS=LinRegGradDesc(X,rings)
Ridge=RidgeReg(data,rings,lam=75)

In [53]:
def RMSE(prediction,actual):
    MSE=np.mean((prediction-actual)**2)
    RMSE=np.sqrt(MSE)
    return RMSE

testData=pd.read_csv('testdata.csv')
testData['gender']=testData['gender'].map({'M':1, 'I':0, 'F':-1})
testArray=np.array(testData.drop('Rings',axis=1))
testRings=np.array(testData['Rings'])
normalisedArray=np.column_stack((np.ones(np.shape(testArray)[0]),(testArray-mu)/sigma))

exactPrediction=normalisedArray@LSExact
gdPrediction=normalisedArray@LSGD
ridgePrediction=normalisedArray@Ridge
exactRMSE=RMSE(exactPrediction,testRings)
GDRMSE=RMSE(gdPrediction,testRings)
RidgeRMSE=RMSE(ridgePrediction,testRings)
print(exactRMSE, "\n", GDRMSE, "\n",RidgeRMSE)
print(LSExact,LSGD,Ridge)


1.970306417199624 
 1.9703049842107359 
 2.001804162553311
[ 6.66024896  0.03053369 -0.82825179  6.02918378  4.49571341  4.08054577
 -9.00214376 -3.98483505  3.77871472] [ 6.65977356  0.0305335  -0.82697649  6.02761979  4.49571999  4.08064011
 -9.00223795 -3.98501406  3.77867729] [ 9.95959641e+00  2.10768410e-03  9.46735917e-01  1.50893338e+00
  1.76263925e+00  1.24367003e+00 -3.37827904e+00  3.94641121e-01
  3.84916260e+00]


In [ ]:
#Bootstrapping

def Bootstrap(X,y,iterations):
    samples=np.shape(X)[0]
    Y=np.column_stack((np.ones(samples),X))
    coefmat1=np.zeros((iterations,np.shape(Y)[1]))
    coefmat2=np.zeros((iterations,np.shape(Y)[1]))
    for i in range(iterations):
        rowchoices=np.random.choice(samples-1,samples,replace=True)
        coefmat1[i]=LinRegExact(Y[rowchoices],y[rowchoices])
        coefmat2[i]=RidgeReg(X[rowchoices],y[rowchoices])
    var1=np.var(coefmat1,axis=0)
    var2=np.var(coefmat2,axis=0)
    mean1=np.mean(coefmat1,axis=0)
    mean2=np.mean(coefmat2,axis=0)
    return var1, var2

print(Bootstrap(data,rings,1000)[2:])

(array([3.16305522e+00, 4.54139655e-04, 9.24178067e-01, 1.53746064e+00,
       8.37134224e+00, 3.42806427e-01, 4.31197171e-01, 7.24076545e-01,
       7.37778561e-01]), array([0.00279224, 0.0004907 , 0.01756794, 0.01227603, 0.02861143,
       0.00654275, 0.02513025, 0.02054383, 0.03504158]))
